----------DAY-8----------

In [1]:
#Import libraries
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

PROGRAM-1

In [2]:
#Use RFE to select top 5 features from House Prices dataset 
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
import random

# Number of houses
n = 30

data = {
    "Area": [],
    "Bedrooms": [],
    "Bathrooms": [],
    "Age": [],
    "Garage": [],
    "Price": []        }

for i in range(n):
    area = random.randint(800, 3500)          # Square feet
    bedrooms = random.randint(1, 5)
    bathrooms = random.randint(1, 4)
    age = random.randint(1, 30)               # Years
    garage = random.randint(0, 3)

    # Generate price based on features
    price = (
        area * 120 +
        bedrooms * 10000 +
        bathrooms * 15000 +
        garage * 8000 -
        age * 2000 +
        random.randint(-20000, 20000)   )

    data["Area"].append(area)
    data["Bedrooms"].append(bedrooms)
    data["Bathrooms"].append(bathrooms)
    data["Age"].append(age)
    data["Garage"].append(garage)
    data["Price"].append(price)

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('House_prices.csv', index=False)
print("CSV file created successfully!")

# Load dataset
df = pd.read_csv("House_prices.csv")

# Select only numerical features
numeric_df = df.select_dtypes(include=['int64', 'float64'])

# Features and target
X = numeric_df.drop('Price' , axis=1)
y = numeric_df['Price']

# Handle missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

# Base model
model = LinearRegression()

# Apply RFE to select top 5 features
rfe = RFE(estimator=model , n_features_to_select=5)
rfe.fit(X_train, y_train)

# Selected features
selected_features = X.columns[rfe.support_]

print("Top 5 Selected Features : ")
for feature in selected_features :
    print(feature)

CSV file created successfully!
Top 5 Selected Features : 
Area
Bedrooms
Bathrooms
Age
Garage


PROGRAM-2

In [3]:
# Create polynomial features (degree=2) for a regression task 
from sklearn.preprocessing import PolynomialFeatures

# Load dataset
df = pd.read_csv("House_prices.csv")

# Features (independent variables)
X = df[['Area', 'Bedrooms', 'Bathrooms', 'Age', 'Garage']]

# Create polynomial features of degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)

X_poly = poly.fit_transform(X)

# Get feature names
feature_names = poly.get_feature_names_out(X.columns)

# Convert to DataFrame
X_poly_df = pd.DataFrame(X_poly, columns=feature_names)

# Display first 5 rows
print(X_poly_df.head())

# Save to CSV (optional)
X_poly_df.to_csv("HousePrices_Polynomial.csv", index=False)

print("\nPolynomial features created successfully!")

     Area  Bedrooms  Bathrooms   Age  Garage      Area^2  Area Bedrooms  \
0  3087.0       2.0        2.0  14.0     3.0   9529569.0         6174.0   
1  3083.0       1.0        1.0  17.0     0.0   9504889.0         3083.0   
2  2186.0       5.0        2.0  20.0     3.0   4778596.0        10930.0   
3  1900.0       3.0        2.0   4.0     1.0   3610000.0         5700.0   
4  3204.0       1.0        2.0  25.0     1.0  10265616.0         3204.0   

   Area Bathrooms  Area Age  Area Garage  Bedrooms^2  Bedrooms Bathrooms  \
0          6174.0   43218.0       9261.0         4.0                 4.0   
1          3083.0   52411.0          0.0         1.0                 1.0   
2          4372.0   43720.0       6558.0        25.0                10.0   
3          3800.0    7600.0       1900.0         9.0                 6.0   
4          6408.0   80100.0       3204.0         1.0                 2.0   

   Bedrooms Age  Bedrooms Garage  Bathrooms^2  Bathrooms Age  \
0          28.0             

PROGRAM-3

In [4]:
#Build a full Pipeline: Imputer → Scaler → Classifier 
# Import libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
cancer = load_breast_cancer()

# Features and target
X = cancer.data
y = cancer.target

# Introduce some missing values (for demonstration)
X[0:20, 0] = np.nan
X[30:40, 3] = np.nan

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

# Build Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train model
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy : ", round(accuracy, 4))
print("\nClassification Report : ")
print(classification_report(y_test, y_pred))

Accuracy :  0.9474

Classification Report : 
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        42
           1       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



PROGRAM-4

In [5]:
#Apply ColumnTransformer for numeric and categorical columns 
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load dataset
df = pd.read_csv("House_prices.csv")

# Features and target
X = df.drop('Price', axis=1)
y = df['Price']

# Numeric and categorical columns
numeric_features = ['Area', 'Bedrooms', 'Age' , 'Bathrooms']
categorical_features = ['Garage']

# Numeric pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Complete pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100 , random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.33 , random_state=42)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Predictions : ", y_pred)
print("\nActual : ", y_test.values)
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error : ", mae)


Predictions :  [205785.73 193739.59 363945.52 386147.15 361999.39 373867.6  365252.85
 385578.31 266514.59 363046.65]

Actual :  [234734 137625 365951 499978 373322 393346 327269 441230 303249 408300]
Mean Absolute Error :  40732.34999999999


--------PRACTICE SHEET----------

PROGRAM-5

In [6]:
#End-to-end pipeline on Titanic: preprocessing + Random Forest 
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load Titanic dataset
df = pd.DataFrame(sns.load_dataset('titanic'))

# Features and target
X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
y = df['survived']

# Numerical and categorical columns
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['pclass', 'sex', 'embarked']

# Numerical pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Complete pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100 , random_state=42))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy : ", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report : ")
print(classification_report(y_test, y_pred))

Accuracy :  0.8156

Classification Report : 
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



PROGRAM-6

In [7]:
#Feature engineering challenge: create 3 new features that improve accuracy 
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Titanic dataset
df = pd.DataFrame(sns.load_dataset("titanic"))

# -----------Feature Engineering-----------

# Family size
df['FamilySize'] = df['sibsp'] + df['parch'] + 1

#  Is passenger alone?
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Fare per person
df['FarePerPerson'] = df['fare'] / df['FamilySize']

# Target
y = df['survived']

# Features
X = df[['pclass' , 'sex', 'age' , 'fare' , 'embarked' , 'FamilySize' , 'IsAlone' , 'FarePerPerson']]

# Numeric and categorical columns
numeric_features = ['age' , 'fare', 'FamilySize' , 'IsAlone' , 'FarePerPerson']
categorical_features = ['pclass' , 'sex' , 'embarked']

# Numeric preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100 , random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy : ", round(accuracy * 100, 2), "%")

Accuracy :  78.77 %
